# Gravité améliorée — Gamma conservé, variables enrichies

Nous gardons la famille **Gamma à lien logarithmique**, avec régularisation. Les trois variables actuelles constituent notre référence, pas une contrainte définitive sur les entrées. Les modèles restent linéaires dans leurs coefficients après transformation des variables ; des splines permettent une relation courbe avec une variable brute.

Le Gamma apprend uniquement sur les montants strictement positifs. Il prédit une gravité conditionnelle pour tous les contrats, ensuite multipliée par la fréquence. La cible reste en euros et aucun sinistre extrême n'est plafonné. Les variantes sont comparées dans cinq plis communs par client. La sélection finale se fait sur les primes, dans `tarification_optimisee.ipynb`.

L’analyse exploratoire complète reste disponible dans [modele_gravite.ipynb](modele_gravite.ipynb). Ce notebook explique les améliorations, leurs comparaisons et le modèle final.

In [1]:
import os
os.environ['OMP_NUM_THREADS']='2'
os.environ['OPENBLAS_NUM_THREADS']='2'
from pathlib import Path
import json,hashlib,time,sys,platform,subprocess,warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cloudpickle,sklearn
from IPython.display import display,Markdown
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler,OneHotEncoder,SplineTransformer,FunctionTransformer
from sklearn.linear_model import LogisticRegression,GammaRegressor
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupShuffleSplit,StratifiedGroupKFold,GroupKFold
from sklearn.metrics import log_loss,brier_score_loss,roc_auc_score,root_mean_squared_error,mean_absolute_error,mean_squared_error
from sklearn.exceptions import ConvergenceWarning
from threadpoolctl import threadpool_limits
threadpool_limits(limits=2)
warnings.filterwarnings('error',category=ConvergenceWarning)
racines=[Path.cwd(),Path.cwd()/'TD-Tarification-automobile-par-Machine-Learning']
ROOT=next(p for p in racines if (p/'Data/train.csv').exists())
BASE_OUTPUT=ROOT/'output/optimisation'
BASE_OUTPUT.mkdir(parents=True,exist_ok=True)
SEED=42
train=pd.read_csv(ROOT/'Data/train.csv',dtype={'code_postal':str})
test=pd.read_csv(ROOT/'Data/test.csv',dtype={'code_postal':str})
ii,jj=next(GroupShuffleSplit(n_splits=1,test_size=.2,random_state=SEED).split(train,groups=train.id_client))
dev=train.iloc[ii].copy();valid=train.iloc[jj].copy()
assert set(dev.id_client).isdisjoint(valid.id_client)
assert train['index'].is_unique and test['index'].is_unique
assert set(train.nombre_sinistres.unique())=={0,1}
assert ((train.nombre_sinistres==0)==(train.montant_sinistre==0)).all()
CV_MAIN=StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=20261009)
MAIN_SPLITS=list(CV_MAIN.split(dev,dev.nombre_sinistres,groups=dev.id_client))
DATA_HASH={name:hashlib.sha256((ROOT/'Data'/name).read_bytes()).hexdigest() for name in ['train.csv','test.csv']}
print('Apprentissage commun :',len(dev),'contrats et',int(dev.nombre_sinistres.sum()),'sinistres.')
print('Diagnostic historique réservé :',len(valid),'contrats ; aucune sélection sur ces cibles.')

Apprentissage commun : 40013 contrats et 2352 sinistres.
Diagnostic historique réservé : 9987 contrats ; aucune sélection sur ces cibles.


## 1. Préparation et variables candidates

Nous testons prix, puissance, âge du conducteur, type de véhicule, département et interactions contrat × usage. Les identifiants et la cible ne sont jamais des prédicteurs. Imputation, standardisation, splines et regroupement des catégories sont appris dans chaque pli. Les codes de département ne sont pas traités comme des valeurs ordonnées.

In [2]:
RAW_FEATURES=['bonus','type_contrat','duree_contrat','anciennete_info','freq_paiement','paiement','utilisation',
    'code_postal','conducteur2','age_conducteur1','age_conducteur2','anciennete_permis1','anciennete_permis2',
    'anciennete_vehicule','cylindre_vehicule','din_vehicule','essence_vehicule','marque_vehicule','modele_vehicule',
    'debut_vente_vehicule','fin_vente_vehicule','vitesse_vehicule','type_vehicule','prix_vehicule','poids_vehicule']
LOW_CATS=['type_contrat','freq_paiement','paiement','utilisation','conducteur2','essence_vehicule','type_vehicule']
NUM_NATIVE=['bonus','duree_contrat','anciennete_info','age_conducteur1','age_conducteur2','anciennete_permis1',
    'anciennete_permis2','anciennete_vehicule','cylindre_vehicule','din_vehicule','prix_vehicule','poids_vehicule',
    'vitesse_vehicule','debut_vente_vehicule','fin_vente_vehicule','puissance_par_tonne']
NUM_COMPACT=['log_bonus','duree_contrat','anciennete_info','age_conducteur1','anciennete_permis2',
    'anciennete_vehicule','log_din','log_poids','log_cylindre']

def preparer_metier(frame):
    needed=['bonus','type_contrat','utilisation','anciennete_vehicule','age_conducteur1',
        'poids_vehicule','din_vehicule','cylindre_vehicule','prix_vehicule','code_postal']
    absent=sorted(set(needed)-set(frame.columns))
    if absent:raise ValueError(f'Colonnes nécessaires absentes : {absent}')
    x=frame.loc[:,[c for c in RAW_FEATURES if c in frame]].copy()
    for col in ['poids_vehicule','cylindre_vehicule']:
        x[col]=x[col].mask(x[col]<=0)
    # Zéros du second conducteur conservés : son absence est une information structurelle.
    s=x.code_postal.astype('string').str.strip().str.upper()
    x['departement']=s.str[:2].where(s.str.fullmatch(r'(?:0[1-9]|1[0-9]|2[1-9]|[3-8][0-9]|9[0-5]|2A|2B)[0-9]{3}',na=False))
    dom=s.str.fullmatch(r'97[1-46][0-9]{2}',na=False)
    x.loc[dom,'departement']=s.loc[dom].str[:3]
    for dest,src in [('log_bonus','bonus'),('log_din','din_vehicule'),('log_poids','poids_vehicule'),
                     ('log_cylindre','cylindre_vehicule'),('log_prix','prix_vehicule')]:
        x[dest]=np.log1p(x[src].mask(x[src]<0))
    x['puissance_par_tonne']=x.din_vehicule/(x.poids_vehicule/1000)
    x['contrat_usage']=x.type_contrat.astype('string').fillna('__NA__')+'|'+x.utilisation.astype('string').fillna('__NA__')
    x['age_maxi']=x.anciennete_vehicule*(x.type_contrat=='Maxi')
    x['prix_maxi']=x.log_prix*(x.type_contrat=='Maxi')
    for col in x.select_dtypes(include=['object','string']).columns:
        x[col]=x[col].astype(object).where(x[col].notna(),np.nan)
    return x

def construire_preparation(numeriques,categorielles,splines=(),min_frequency=50):
    splines=list(splines)
    lineaires=[c for c in numeriques if c not in splines]
    parts=[]
    if lineaires:
        parts.append(('lineaire',Pipeline([('imputation',SimpleImputer(strategy='median',add_indicator=True)),
            ('echelle',StandardScaler())]),lineaires))
    if splines:
        parts.append(('courbes',Pipeline([('imputation',SimpleImputer(strategy='median')),
            ('splines',SplineTransformer(n_knots=4,degree=2,knots='quantile',include_bias=False,extrapolation='constant')),
            ('echelle',StandardScaler())]),splines))
    if categorielles:
        parts.append(('categories',Pipeline([('imputation',SimpleImputer(strategy='constant',fill_value='__NA__')),
            ('encodage',OneHotEncoder(min_frequency=min_frequency,handle_unknown='infrequent_if_exist',sparse_output=False))]),list(categorielles)))
    return ColumnTransformer(parts,remainder='drop')

## 2. Gamma linéaire et variantes métier

Le modèle calcule `exp(intercept + coefficients × entrées transformées)` et conserve des sorties positives. Gamma optimise sa déviance régularisée ; la sélection évaluera la RMSE de prime.

L'essai « coût du modèle de voiture » calcule une moyenne de dommage lissée vers la moyenne d'apprentissage. Pour entraîner Gamma, cette moyenne est calculée dans les autres plis **par client**, avec un prior lui-même hors pli. Pour prédire, les catégories inconnues reçoivent la moyenne de l'apprentissage. Chaque validation externe reste entièrement exclue de ces calculs. Cela répond à la proposition d'encodage métier sans laisser la ligne voir son propre dommage.

La régularisation de la référence à trois variables est également comparée sur une plage étendue : alpha = 0,001 / 0,01 / 0,1 / 1 / 10 / 100. Cela permet de tester un retour progressif vers une gravité presque constante dans le critère de prime.

In [3]:

def cache_prediction(X,y,Xeval,config,component,fit_predict):
    signature=hashlib.sha256(CACHE_SIGNATURE.encode())
    signature.update(np.asarray(X['index'],dtype=np.int64).tobytes())
    signature.update(np.asarray(Xeval['index'],dtype=np.int64).tobytes())
    signature.update(np.asarray(y,dtype=np.float64).tobytes())
    signature.update(json.dumps(config,sort_keys=True).encode())
    cache_dir=BASE_OUTPUT/'cache'/component
    cache_dir.mkdir(parents=True,exist_ok=True)
    target=cache_dir/(signature.hexdigest()+'.npy')
    if target.exists():
        result=np.load(target,allow_pickle=False)
        if result.shape==(len(Xeval),) and np.isfinite(result).all():return result
    started=time.perf_counter()
    result=np.asarray(fit_predict(),dtype=float)
    assert result.shape==(len(Xeval),) and np.isfinite(result).all()
    temporary=target.with_suffix('.tmp')
    with temporary.open('wb') as f:np.save(f,result,allow_pickle=False)
    temporary.replace(target)
    print(component,config['name'],':',round(time.perf_counter()-started,1),'s',flush=True)
    return result

class GammaMetier:
    def __init__(self,config):self.config=config
    def _spec(self):
        c=self.config
        nums=['anciennete_vehicule'];cats=['type_contrat','utilisation']
        if c.get('compact'):nums+=['log_prix','age_conducteur1','log_din'];cats+=['conducteur2','type_vehicule']
        if c.get('interaction'):nums+=['age_maxi','prix_maxi'];cats+=['contrat_usage']
        if c.get('geo'):cats+=['departement']
        if c.get('vehicules'):cats+=['marque_vehicule','modele_vehicule']
        if c.get('target'):nums+=['cout_modele_lisse','cout_marque_lisse']
        splines=['anciennete_vehicule','log_prix','age_conducteur1'] if c.get('splines') else []
        return nums,cats,splines
    def _mappings(self,X,y,prior):
        maps={}
        for col in ['modele_vehicule','marque_vehicule']:
            stats=pd.Series(np.asarray(y),index=X.index).groupby(X[col].fillna('__NA__')).agg(['sum','count'])
            maps[col]=(stats['sum']+self.config['lissage']*prior)/(stats['count']+self.config['lissage'])
        return maps
    def _encode(self,X,maps,prior):
        out=pd.DataFrame(index=X.index)
        for col,dest in [('modele_vehicule','cout_modele_lisse'),('marque_vehicule','cout_marque_lisse')]:
            out[dest]=X[col].fillna('__NA__').map(maps[col]).fillna(prior)
        return out
    def fit(self,X,y,groups):
        y=np.asarray(y,dtype=float);groups=np.asarray(groups)
        assert (y>0).all() and len(X)==len(groups)
        x=preparer_metier(X)
        if self.config.get('target'):
            self.prior_=float(y.mean());self.maps_=self._mappings(x,y,self.prior_)
            enc=pd.DataFrame(np.nan,index=x.index,columns=['cout_modele_lisse','cout_marque_lisse'])
            inner=GroupKFold(n_splits=3,shuffle=True,random_state=2303)
            for a,b in inner.split(x,y,groups=groups):
                assert set(groups[a]).isdisjoint(groups[b])
                prior=float(y[a].mean());maps=self._mappings(x.iloc[a],y[a],prior)
                enc.loc[x.index[b]]=self._encode(x.iloc[b],maps,prior).to_numpy()
            x=x.join(enc)
        nums,cats,splines=self._spec()
        self.prep_=construire_preparation(nums,cats,splines,self.config.get('rare',20))
        z=self.prep_.fit_transform(x)
        self.model_=GammaRegressor(alpha=self.config['alpha'],solver='lbfgs',max_iter=5000,tol=1e-5)
        self.model_.fit(z,y)
        return self
    def predict(self,X):
        x=preparer_metier(X)
        if self.config.get('target'):x=x.join(self._encode(x,self.maps_,self.prior_))
        p=self.model_.predict(self.prep_.transform(x))
        if not np.isfinite(p).all() or not (p>0).all():raise ValueError('Montants Gamma invalides.')
        return p

def creer_gravite(config):return GammaMetier(config)

GRAVITES=[{'name':'Gamma_reference_3_variables','alpha':.1}]
for alpha in [.001,.01,1.,10.,100.]:
    GRAVITES.append({'name':f'Gamma_3_variables_a{alpha}','alpha':alpha})
for label,kwargs in [
    ('compact',{'compact':True}),
    ('splines',{'compact':True,'splines':True}),
    ('interactions',{'compact':True,'splines':True,'interaction':True}),
    ('departement',{'compact':True,'splines':True,'geo':True}),
]:
    for alpha in [.01,.1,1.]:GRAVITES.append({'name':f'Gamma_{label}_a{alpha}','alpha':alpha,**kwargs})
for smoothing in [20,100]:
    for alpha in [.1,1.]:GRAVITES.append({'name':f'Gamma_cout_modele_lisse{ smoothing }_a{alpha}',
        'alpha':alpha,'compact':True,'target':True,'lissage':smoothing})
GRAVITES.append({'name':'Gamma_vehicules_regroupees','alpha':1.,'compact':True,'vehicules':True,'rare':50})

def ajuster_gravites(X,y,groups,Xeval,configs):
    values=[]
    for c in configs:
        values.append(cache_prediction(X,y,Xeval,c,'gravite',
            lambda c=c: creer_gravite(c).fit(X,y,groups).predict(Xeval)))
    p=np.column_stack(values)
    assert np.isfinite(p).all() and (p>0).all()
    return p
print(len(GRAVITES),'candidats Gamma : la famille et le lien log sont conservés.')

23 candidats Gamma : la famille et le lien log sont conservés.


## 3. Comparaison hors pli

Les montants sont prédits pour tout le développement afin d'assembler les primes sur les mêmes contrats. La RMSE et la MAE conditionnelles du tableau ci-dessous ne portent que sur les sinistrés. Le gagnant conditionnel peut différer de la meilleure composante du produit final.

Les prédictions intermédiaires sont mises en cache avec une empreinte du code, des données, des identifiants, des cibles d’apprentissage et des paramètres. Une reprise ne réutilise que des calculs identiques ; aucun score historique ne sert à décider quels plis conserver.

In [4]:
import nbformat
_source_nb=nbformat.read(ROOT/'modele_gravite_ameliore.ipynb',as_version=4)
_source_code='\n'.join(c.source for c in _source_nb.cells[:7] if c.cell_type=='code')
CACHE_SIGNATURE=hashlib.sha256((_source_code+json.dumps(DATA_HASH,sort_keys=True)+sklearn.__version__).encode()).hexdigest()
OUT=BASE_OUTPUT/'gravite';OUT.mkdir(parents=True,exist_ok=True)
names=[c['name'] for c in GRAVITES]
oof=np.full((len(dev),len(names)),np.nan);fold_rows=[]
started=time.perf_counter()
for fold,(a,b) in enumerate(MAIN_SPLITS,1):
    learn=dev.iloc[a];claims=learn.loc[learn.nombre_sinistres.eq(1)]
    assert set(claims.id_client).isdisjoint(dev.iloc[b].id_client)
    p=ajuster_gravites(claims,claims.montant_sinistre,claims.id_client,dev.iloc[b],GRAVITES)
    oof[b]=p
    mask=dev.iloc[b].nombre_sinistres.eq(1).to_numpy()
    for j,name in enumerate(names):
        fold_rows.append({'pli':fold,'modele':name,'RMSE_sinistres':root_mean_squared_error(dev.iloc[b].loc[mask,'montant_sinistre'],p[mask,j])})
    (BASE_OUTPUT/'progression_gravite.json').write_text(json.dumps({'etape':'CV gravite','pli':fold,'plis':5,'secondes':time.perf_counter()-started}))
    print('Gravité : pli',fold,'/ 5 terminé.',flush=True)
assert np.isfinite(oof).all() and (oof>0).all()
np.savez_compressed(OUT/'predictions_oof.npz',ids=dev['index'].to_numpy(),predictions=oof,names=np.array(names))
pd.DataFrame(fold_rows).to_csv(OUT/'scores_plis.csv',index=False)
mask=dev.nombre_sinistres.eq(1).to_numpy()
table=pd.DataFrame([{'modele':name,'RMSE_OOF_sinistres':root_mean_squared_error(dev.loc[mask,'montant_sinistre'],oof[mask,j]),
    'MAE_OOF_sinistres':mean_absolute_error(dev.loc[mask,'montant_sinistre'],oof[mask,j])}
    for j,name in enumerate(names)]).sort_values('RMSE_OOF_sinistres')
table.to_csv(OUT/'comparaison_cv.csv',index=False)
(OUT/'catalogue.json').write_text(json.dumps({'configs':GRAVITES,'sha256_data':DATA_HASH},ensure_ascii=False,indent=2))
with (OUT/'fabrique.pkl').open('wb') as f:cloudpickle.dump({'make':creer_gravite,'batch':ajuster_gravites,'configs':GRAVITES},f)
display(table)
print('Meilleure gravité en RMSE conditionnelle seule :',table.iloc[0].modele)
print('La paire finale sera choisie avec les probabilités de fréquence sur tous les contrats.')

gravite Gamma_reference_3_variables : 0.1 s
gravite Gamma_3_variables_a0.001 : 0.2 s
gravite Gamma_3_variables_a0.01 : 0.1 s
gravite Gamma_3_variables_a1.0 : 0.2 s
gravite Gamma_3_variables_a10.0 : 0.1 s
gravite Gamma_3_variables_a100.0 : 0.1 s
gravite Gamma_compact_a0.01 : 0.1 s
gravite Gamma_compact_a0.1 : 0.2 s
gravite Gamma_compact_a1.0 : 0.1 s
gravite Gamma_splines_a0.01 : 0.2 s
gravite Gamma_splines_a0.1 : 0.1 s
gravite Gamma_splines_a1.0 : 0.2 s
gravite Gamma_interactions_a0.01 : 0.3 s
gravite Gamma_interactions_a0.1 : 0.2 s
gravite Gamma_interactions_a1.0 : 0.1 s
gravite Gamma_departement_a0.01 : 0.2 s
gravite Gamma_departement_a0.1 : 0.3 s
gravite Gamma_departement_a1.0 : 0.2 s
gravite Gamma_cout_modele_lisse20_a0.1 : 0.4 s
gravite Gamma_cout_modele_lisse20_a1.0 : 0.4 s
gravite Gamma_cout_modele_lisse100_a0.1 : 0.4 s
gravite Gamma_cout_modele_lisse100_a1.0 : 0.4 s
gravite Gamma_vehicules_regroupees : 0.2 s
Gravité : pli 1 / 5 terminé.
gravite Gamma_reference_3_variables : 0.2 

,modele,RMSE_OOF_sinistres,MAE_OOF_sinistres
0,Gamma_reference_3_variables,1499.952599,932.621174
2,Gamma_3_variables_a0.01,1501.248062,932.856778
1,Gamma_3_variables_a0.001,1501.719101,933.198409
3,Gamma_3_variables_a1.0,1502.597611,940.486484
7,Gamma_compact_a0.1,1502.624209,934.131598
20,Gamma_cout_modele_lisse100_a0.1,1502.794928,934.487379
14,Gamma_interactions_a1.0,1503.302891,936.236340
8,Gamma_compact_a1.0,1503.579375,940.746756
21,Gamma_cout_modele_lisse100_a1.0,1503.579981,940.832369
17,Gamma_departement_a1.0,1503.692251,939.152529


## 4. Modèle final retenu

Cette cellule affiche la configuration effectivement sélectionnée pour les primes après l'assemblage. Le modèle final est réentraîné sur tous les sinistres positifs du train. Les trois variables initiales et la configuration initiale restent des références vérifiables dans le notebook précédent.

### Comprendre le modèle retenu

Le Gamma à lien log estime **E[montant | sinistre, caractéristiques]**, avec `montant prédit = exp(intercept + somme des coefficients × variables préparées)`. L'exponentielle garantit un montant positif. L'apprentissage optimise la déviance Gamma avec une pénalisation L2, contrôlée par `alpha` ; la sélection utilise la RMSE des **primes combinées**, qui peut préférer un autre alpha que la seule RMSE conditionnelle.

Les candidats ont testé davantage de variables, y compris le département et le coût moyen lissé par modèle de voiture, appris hors client. Si trois variables restent retenues, c'est le résultat de la comparaison, pas une règle imposée. Dans le tableau des coefficients ci-dessous, un coefficient numérique concerne une unité **après transformation et standardisation**. Pour comparer deux modalités d'une même variable, leur facteur relatif est `exp(coefficient A − coefficient B)`, toutes choses égales par ailleurs ; ces associations ne sont pas des effets causaux.

Le modèle apprend uniquement sur les sinistres positifs du train. Il prédit ensuite un montant **conditionnel** pour chaque prospect, car nous ne connaissons pas encore ceux qui auront un accident. Le modèle de fréquence fournit la probabilité à multiplier. Le test n'est pas une validation et ne permet pas de calculer une RMSE sans ses cibles.

In [5]:
COMPONENT='gravite'
PICKLE_NAME='modele_gravite_final.pkl'

config_path=BASE_OUTPUT/'prime/configuration_finale.json'
if config_path.exists():
    final_config=json.loads(config_path.read_text())
    choix=final_config[COMPONENT]
    display(Markdown('## MODÈLE FINAL RETENU POUR LA PRIME\n\n**'+choix['name']+'**\n\nParamètres : `'+json.dumps(choix,ensure_ascii=False)+'`'))
    with (BASE_OUTPUT/'prime'/PICKLE_NAME).open('rb') as f:modele_final=cloudpickle.load(f)
    sorties=modele_final.predict_proba(test.iloc[:5])[:,1] if COMPONENT=='frequence' else modele_final.predict(test.iloc[:5])
    display(pd.DataFrame({'index':test['index'].iloc[:5].to_numpy(),'prediction_composante':sorties}))

    nums,cats,splines=modele_final._spec()
    display(Markdown('Variables numériques retenues : **'+', '.join(nums)+'**. Variables catégorielles : **'+', '.join(cats)+'**.'))
    noms=modele_final.prep_.get_feature_names_out()
    coefficients=pd.DataFrame({'entree_preparee':noms,'coefficient_log':modele_final.model_.coef_})
    coefficients['exp_coefficient']=np.exp(coefficients.coefficient_log)
    coefficients.to_csv(BASE_OUTPUT/'gravite/coefficients_modele_final.csv',index=False)
    display(coefficients)
    print('Intercept sur échelle log :',modele_final.model_.intercept_)
    print('Alpha retenu :',choix['alpha'])
else:
    print('Catalogue prêt. Exécuter tarification_optimisee.ipynb pour choisir la paire et entraîner la version finale.')

Intercept sur échelle log : 7.376915680275907
Alpha retenu : 0.01


## MODÈLE FINAL RETENU POUR LA PRIME

**Gamma_3_variables_a0.01**

Paramètres : `{"name": "Gamma_3_variables_a0.01", "alpha": 0.01}`

,index,prediction_composante
0,50000,1508.877797
1,50001,1243.392018
2,50002,1963.278784
3,50003,1866.793126
4,50004,1502.245078


Variables numériques retenues : **anciennete_vehicule**. Variables catégorielles : **type_contrat, utilisation**.

,entree_preparee,coefficient_log,exp_coefficient
0,lineaire__anciennete_vehicule,-0.060875,0.940941
1,categories__type_contrat_Maxi,0.109408,1.115618
2,categories__type_contrat_Median1,0.005533,1.005548
3,categories__type_contrat_Median2,-0.096713,0.907816
4,categories__type_contrat_Mini,-0.018096,0.982067
5,categories__utilisation_Professional,0.065553,1.067749
6,categories__utilisation_Retired,-0.051112,0.950172
7,categories__utilisation_WorkPrivate,0.035759,1.036406
8,categories__utilisation_infrequent_sklearn,-0.050068,0.951165


Références : [GammaRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.GammaRegressor.html), [SplineTransformer](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.SplineTransformer.html). Avec 2 917 sinistres, des catégories fines peuvent rester instables ; l'encodage cible et la complexité ne sont adoptés que s'ils sont sélectionnés dans le protocole commun.